# Lab 0b · Hugging Face — the full model cycle

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omar-florez/training-efficient-llms/blob/main/labs/lab_huggingface.ipynb)

One notebook, one complete lap around the track this book keeps circling: **load data → tokenize → load a model → train → generate → evaluate → save.** The model is `distilgpt2` (82M parameters — the same architecture as Chapter 3, small enough to fine-tune in minutes) and the data is WikiText-2. In Colab, switch on a GPU first: *Runtime → Change runtime type → T4 GPU*.

📖 Companion program: [Training Efficient LLMs](https://omar-florez.github.io/training-efficient-llms/)

In [ ]:
%pip -q install transformers datasets accelerate
import torch, transformers, math
print("transformers", transformers.__version__, "| GPU:", torch.cuda.is_available())

## 1 · Load data

`datasets` downloads, caches, and memory-maps a corpus in one call. We take a slice so the whole notebook runs in minutes; the pattern is identical for a terabyte (Chapters 5–7 are about what *goes into* this call).

In [ ]:
from datasets import load_dataset

raw = load_dataset("wikitext", "wikitext-2-raw-v1")
print(raw)                                  # train / validation / test splits
train_txt = raw["train"].select(range(4000))    # a small slice for speed
val_txt   = raw["validation"].select(range(400))
print("\nexample row:\n", train_txt[9]["text"][:300])

## 2 · Tokenize

The model never sees text — it sees integer ids (Chapter 2). The tokenizer that made the model must be the one you use: it ships with the checkpoint.

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("distilgpt2")
tok.pad_token = tok.eos_token               # GPT-2 has no pad token; reuse end-of-text

s = "Efficient training of language models"
print("tokens:", tok.tokenize(s))
print("ids   :", tok(s)["input_ids"])
print("round trip:", tok.decode(tok(s)["input_ids"]))

In [ ]:
# tokenize the corpus, then pack ids into fixed 128-token blocks (Chapter 16 does this at scale)
def tokenize(batch): return tok(batch["text"])
tokenized = train_txt.map(tokenize, batched=True, remove_columns=["text"])
val_tok   = val_txt.map(tokenize, batched=True, remove_columns=["text"])

BLOCK = 128
def group(examples):
    ids = sum(examples["input_ids"], [])                 # concatenate everything
    n = len(ids) // BLOCK * BLOCK
    chunks = [ids[i:i+BLOCK] for i in range(0, n, BLOCK)]
    return {"input_ids": chunks, "labels": [c[:] for c in chunks]}

lm_train = tokenized.map(group, batched=True, remove_columns=tokenized.column_names)
lm_val   = val_tok.map(group, batched=True, remove_columns=val_tok.column_names)
print(len(lm_train), "training blocks of", BLOCK, "tokens")

## 3 · Load the model — and see what it says *before* training

`AutoModelForCausalLM` gives you the transformer plus the language-model head, weights included. Generating first gives us a baseline to compare against after fine-tuning.

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("distilgpt2")
print(f"parameters: {model.num_parameters():,}")        # ~82M

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

def generate(prompt, n=40):
    ids = tok(prompt, return_tensors="pt").to(device)
    out = model.generate(**ids, max_new_tokens=n, do_sample=True, temperature=0.8,
                         top_p=0.95, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0], skip_special_tokens=True)

print("BEFORE:\n", generate("The history of Peru begins"))

## 4 · Train

`Trainer` wraps the forward/loss/backward/update loop from Lab 0a with batching, mixed precision, logging, and checkpointing. The data collator stacks blocks into batches; for causal LM the labels are the inputs shifted by one (Chapter 2), which the model handles internally.

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

args = TrainingArguments(
    output_dir="distilgpt2-wikitext",
    per_device_train_batch_size=16,
    num_train_epochs=1,
    learning_rate=5e-5,
    logging_steps=20,
    eval_strategy="no",
    report_to="none",
    fp16=torch.cuda.is_available(),
)
collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm=False)
trainer = Trainer(model=model, args=args, train_dataset=lm_train, data_collator=collator)
result = trainer.train()
print(result.metrics)

In [ ]:
import matplotlib.pyplot as plt
NAVY, AMBER = "#17406b", "#b45309"
logs = [(h["step"], h["loss"]) for h in trainer.state.log_history if "loss" in h]
steps, losses = zip(*logs)
plt.figure(figsize=(7, 3.5))
plt.plot(steps, losses, color=NAVY, lw=2)
plt.xlabel("step"); plt.ylabel("cross-entropy loss"); plt.title("Fine-tuning distilgpt2 on WikiText-2")
plt.grid(color="#e3eaf3"); plt.gca().spines[["top","right"]].set_visible(False)
plt.show()

## 5 · Inference after training

Same prompt as before. One epoch on 4,000 articles won't transform an 82M model, but you should see it drift toward Wikipedia's register — flatter tone, encyclopedic phrasing.

In [ ]:
print("AFTER:\n", generate("The history of Peru begins"))

## 6 · Evaluate

The honest number is **perplexity on held-out text** (Chapter 2): exponentiate the average cross-entropy on data the model never trained on. GPT-2-family models score ≈ 29–40 on WikiText-2 depending on size; watch where our fine-tuned distilgpt2 lands.

In [ ]:
eval_metrics = trainer.evaluate(eval_dataset=lm_val)
ppl = math.exp(eval_metrics["eval_loss"])
print(f"validation loss {eval_metrics['eval_loss']:.3f}  ->  perplexity {ppl:.1f}")

## 7 · Save, reload, ship

`save_pretrained` writes weights + config + tokenizer to a folder; `from_pretrained` on that folder (or a Hub repo name) brings it back anywhere. `pipeline` is the one-liner deployment wrapper.

In [ ]:
from transformers import pipeline

model.save_pretrained("my-distilgpt2");  tok.save_pretrained("my-distilgpt2")

gen = pipeline("text-generation", model="my-distilgpt2", device=0 if device=="cuda" else -1)
print(gen("Language models are trained", max_new_tokens=30, do_sample=True)[0]["generated_text"])

## Exercises

1. **Data ablation**: train on 1,000 blocks instead of all of them. How much worse is perplexity? (This is a tiny scaling law — Chapter 4.)
2. **Learning rate**: rerun with `learning_rate=5e-3`. Watch the loss curve do what Chapter 1's divergence figure predicts.
3. **Longer blocks**: set `BLOCK = 512`. Batch size must shrink to fit memory — why? (Activation memory, Chapter 15.)
4. **Better model**: swap `distilgpt2` for `gpt2-medium` (355M). Compare perplexity per minute of training — efficiency is *the* theme of this book.
5. **Prompt the base vs. tuned model** with the first sentence of a WikiText validation article and compare continuations to the real article.